# Delta Lake Concurrency

This notebook demonstrates Delta Lake's concurrency control features. We'll:

1. Simulate concurrent batch and streaming operations
2. Demonstrate optimistic concurrency control
3. Show how Delta Lake handles conflicts
4. Explore transaction isolation levels
5. Demonstrate concurrent reads and writes

## 1. Initialize Spark Session with Delta Lake

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import threading
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_date, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType

# Set SPARK_HOME environment variable
os.environ['SPARK_HOME'] = '/usr/local/lib/python3.10/site-packages/pyspark'

# Add scripts directory to path
sys.path.append('/opt/spark/scripts')
try:
    import utils
except ImportError:
    print("Could not import utils module")

# Create Spark session with Delta Lake support
spark = utils.create_spark_session("Delta Lake Concurrency")

print(f"Spark version: {spark.version}")
try:
    delta_version = spark.sql("SELECT version() as delta_version").collect()[0][0]
    print(f"Delta Lake version: {delta_version}")
except:
    print("Could not determine Delta Lake version")

## 2. Define Constants and Paths

In [ ]:
# Define paths
DATA_DIR = "/opt/spark/data"
DELTA_TABLE_PATH = os.path.join(DATA_DIR, "processed/global_superstore_delta")
CONCURRENCY_TEST_PATH = os.path.join(DATA_DIR, "concurrency_test")

# Create directories if they don't exist
os.makedirs(CONCURRENCY_TEST_PATH, exist_ok=True)

print(f"Delta table path: {DELTA_TABLE_PATH}")
print(f"Concurrency test path: {CONCURRENCY_TEST_PATH}")

## 3. Create a Test Delta Table

In [ ]:
# Generate test data
test_data = utils.generate_test_data(num_records=100, scenario='normal')
test_df = spark.createDataFrame(test_data)

# Write to a test Delta table
test_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(CONCURRENCY_TEST_PATH)

print(f"Created test Delta table at {CONCURRENCY_TEST_PATH} with {test_df.count()} records")

## 4. Simulate Concurrent Updates

In [ ]:
# Define update functions
def update_sales(order_id, new_sales):
    """Update sales for a specific order"""
    try:
        spark.sql(f"""
            UPDATE delta.`{CONCURRENCY_TEST_PATH}`
            SET Sales = {new_sales}
            WHERE `Order ID` = '{order_id}'
        """)
        print(f"Updated Sales to {new_sales} for Order ID {order_id}")
        return True
    except Exception as e:
        print(f"Error updating Sales for Order ID {order_id}: {e}")
        return False

def update_profit(order_id, new_profit):
    """Update profit for a specific order"""
    try:
        spark.sql(f"""
            UPDATE delta.`{CONCURRENCY_TEST_PATH}`
            SET Profit = {new_profit}
            WHERE `Order ID` = '{order_id}'
        """)
        print(f"Updated Profit to {new_profit} for Order ID {order_id}")
        return True
    except Exception as e:
        print(f"Error updating Profit for Order ID {order_id}: {e}")
        return False

# Get a sample order ID
sample_order = spark.read.format("delta").load(CONCURRENCY_TEST_PATH).select("Order ID").limit(1).collect()[0][0]
print(f"Sample Order ID: {sample_order}")

# Simulate concurrent updates
print("\nSimulating concurrent updates...")
update_sales(sample_order, 1000.0)
update_profit(sample_order, 500.0)

# Check the results
print("\nChecking results:")
spark.read.format("delta").load(CONCURRENCY_TEST_PATH).filter(col("Order ID") == sample_order).show()

## 5. Demonstrate Optimistic Concurrency Control

In [ ]:
# Define a function to simulate concurrent transactions
def run_concurrent_updates():
    """Run concurrent updates to demonstrate optimistic concurrency control"""
    # Get a sample order ID
    sample_orders = spark.read.format("delta").load(CONCURRENCY_TEST_PATH).select("Order ID").limit(5).collect()
    sample_order_ids = [row[0] for row in sample_orders]
    
    # Define update functions for threads
    def update_thread_1():
        for order_id in sample_order_ids:
            update_sales(order_id, np.random.uniform(1000, 2000))
            time.sleep(0.5)  # Small delay to increase chance of conflict
    
    def update_thread_2():
        for order_id in sample_order_ids:
            update_profit(order_id, np.random.uniform(500, 1000))
            time.sleep(0.5)  # Small delay to increase chance of conflict
    
    # Create and start threads
    thread1 = threading.Thread(target=update_thread_1)
    thread2 = threading.Thread(target=update_thread_2)
    
    thread1.start()
    thread2.start()
    
    # Wait for threads to complete
    thread1.join()
    thread2.join()
    
    # Check the results
    print("\nFinal state after concurrent updates:")
    spark.read.format("delta").load(CONCURRENCY_TEST_PATH).filter(col("Order ID").isin(sample_order_ids)).show()

# Run concurrent updates
print("Running concurrent updates to demonstrate optimistic concurrency control...")
run_concurrent_updates()

## 6. Explore Transaction Isolation Levels

In [ ]:
# Demonstrate Serializable isolation
print("Demonstrating Serializable isolation...")

# Start a transaction
spark.sql("START TRANSACTION")

# Read data
initial_count = spark.sql(f"SELECT COUNT(*) FROM delta.`{CONCURRENCY_TEST_PATH}`").collect()[0][0]
print(f"Initial count in transaction: {initial_count}")

# In another session (simulated), add new data
new_data = utils.generate_test_data(num_records=10, scenario='normal')
new_df = spark.createDataFrame(new_data)
new_df.write.format("delta").mode("append").save(CONCURRENCY_TEST_PATH)
print(f"Added 10 new records outside the transaction")

# Read data again in the transaction
transaction_count = spark.sql(f"SELECT COUNT(*) FROM delta.`{CONCURRENCY_TEST_PATH}`").collect()[0][0]
print(f"Count in transaction after external append: {transaction_count}")

# Commit the transaction
spark.sql("COMMIT")

# Read data after commit
final_count = spark.sql(f"SELECT COUNT(*) FROM delta.`{CONCURRENCY_TEST_PATH}`").collect()[0][0]
print(f"Final count after commit: {final_count}")

## 7. Demonstrate Concurrent Reads and Writes

In [ ]:
# Define a function to simulate concurrent reads and writes
def run_concurrent_reads_writes():
    """Run concurrent reads and writes to demonstrate Delta Lake's ACID properties"""
    # Define read function
    def read_thread():
        for i in range(5):
            count = spark.read.format("delta").load(CONCURRENCY_TEST_PATH).count()
            print(f"Read thread iteration {i+1}: Count = {count}")
            time.sleep(1)
    
    # Define write function
    def write_thread():
        for i in range(3):
            new_data = utils.generate_test_data(num_records=5, scenario='normal')
            new_df = spark.createDataFrame(new_data)
            new_df.write.format("delta").mode("append").save(CONCURRENCY_TEST_PATH)
            print(f"Write thread iteration {i+1}: Added 5 records")
            time.sleep(2)
    
    # Create and start threads
    read_thread_obj = threading.Thread(target=read_thread)
    write_thread_obj = threading.Thread(target=write_thread)
    
    read_thread_obj.start()
    write_thread_obj.start()
    
    # Wait for threads to complete
    read_thread_obj.join()
    write_thread_obj.join()
    
    # Final count
    final_count = spark.read.format("delta").load(CONCURRENCY_TEST_PATH).count()
    print(f"\nFinal count after concurrent reads and writes: {final_count}")

# Run concurrent reads and writes
print("Running concurrent reads and writes to demonstrate Delta Lake's ACID properties...")
run_concurrent_reads_writes()

## 8. Explore Delta Table History

In [ ]:
# Get Delta table history
print("Delta Table History:")
spark.sql(f"DESCRIBE HISTORY delta.`{CONCURRENCY_TEST_PATH}`").show(truncate=False)

## 9. Concurrency Demo Complete

In [ ]:
print("Delta Lake concurrency demo completed successfully!")
print(f"Test Delta table is available at: {CONCURRENCY_TEST_PATH}")
print("You can now proceed with the other notebooks in the demo.")